In [ ]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

# 🧹 Jalon 1 : Data Wrangling & Nettoyage (Squelette Étudiant)

Ce notebook correspond à la première étape du **Jalon 1**. L'objectif est d'importer le jeu de données brut (`data/raw/raw_data_sample.csv`), d'effectuer un audit de sa qualité (données manquantes, anomalies physiques, formats de dates hétérogènes) et de le nettoyer à l'aide de votre package personnalisé `src.data_clean`.

### 1. Importation des packages et chargement des données

In [2]:
import os
import sys
import pandas as pd
import numpy as np

# Ajout du dossier parent au chemin de recherche de modules pour importer 'src'
sys.path.append(os.path.abspath('..'))
from src import data_clean as dc

print("Libraries importées avec succès ! Prêt à démarrer le Wrangling.")

Libraries importées avec succès ! Prêt à démarrer le Wrangling.


In [ ]:
raw_data_path = "../data/raw/owid-monkeypox-data.csv"

df_raw = pd.read_csv(raw_data_path)

df_raw.head()

,location,iso_code,date,total_cases,total_deaths,new_cases,new_deaths,new_cases_smoothed,new_deaths_smoothed,new_cases_per_million,total_cases_per_million,new_cases_smoothed_per_million,new_deaths_per_million,total_deaths_per_million,new_deaths_smoothed_per_million
0,Africa,OWID_AFR,2022-05-01,27.0,2.0,0.0,0.0,0.29,NaN,NaN,NaN,NaN,NaN,0.0014,0.0
1,Africa,OWID_AFR,2022-05-02,27.0,2.0,0.0,0.0,0.29,NaN,NaN,0.019,NaN,0.0,0.0014,0.0
2,Africa,OWID_AFR,2022-05-03,27.0,2.0,0.0,0.0,0.29,0.0,NaN,NaN,0.0,NaN,0.0014,0.0
3,Africa,OWID_AFR,2022-05-04,27.0,2.0,0.0,0.0,0.29,NaN,0.0,0.019,NaN,0.0,0.0014,0.0
4,Africa,OWID_AFR,2022-05-05,27.0,2.0,0.0,0.0,0.29,NaN,0.0,NaN,NaN,NaN,0.0014,0.0


### 2. Audit initial des données

**À faire par l'étudiant :**
Explorez le dataset brut pour évaluer sa structure :
- Quelles sont les dimensions du dataset ?
- Quels sont les types de données par colonne ?
- Reste-t-il des valeurs nulles ? Quel est le taux de valeurs manquantes par variable ?
- Y a-t-il des doublons ?

In [39]:
print("Dimensions :", df_raw.shape)
df_raw.info()

Dimensions : (33669, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33669 entries, 0 to 33668
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   location                         33669 non-null  object 
 1   iso_code                         33669 non-null  object 
 2   date                             33669 non-null  object 
 3   total_cases                      33669 non-null  float64
 4   total_deaths                     33663 non-null  float64
 5   new_cases                        33663 non-null  float64
 6   new_deaths                       33666 non-null  float64
 7   new_cases_smoothed               33669 non-null  float64
 8   new_deaths_smoothed              13391 non-null  float64
 9   new_cases_per_million            13494 non-null  float64
 10  total_cases_per_million          13484 non-null  float64
 11  new_cases_smoothed_per_million   13408 non-null  float6

In [40]:
print("Valeurs manquantes par colonne :")
df_raw.isnull().sum()

Valeurs manquantes par colonne :


location                               0
iso_code                               0
date                                   0
total_cases                            0
total_deaths                           6
new_cases                              6
new_deaths                             3
new_cases_smoothed                     0
new_deaths_smoothed                20278
new_cases_per_million              20175
total_cases_per_million            20185
new_cases_smoothed_per_million     20261
new_deaths_per_million             20063
total_deaths_per_million               0
new_deaths_smoothed_per_million        0
dtype: int64

In [41]:
print("Taux de valeurs manquantes (%) :")
(df_raw.isnull().mean() * 100).sort_values(ascending=False)

Taux de valeurs manquantes (%) :


new_deaths_smoothed                60.227509
new_cases_smoothed_per_million     60.177017
total_cases_per_million            59.951291
new_cases_per_million              59.921590
new_deaths_per_million             59.588939
new_cases                           0.017821
total_deaths                        0.017821
new_deaths                          0.008910
location                            0.000000
iso_code                            0.000000
date                                0.000000
total_cases                         0.000000
new_cases_smoothed                  0.000000
total_deaths_per_million            0.000000
new_deaths_smoothed_per_million     0.000000
dtype: float64

In [42]:
print("Nombre de doublons :", df_raw.duplicated().sum())

Nombre de doublons : 3


In [43]:
df_raw.describe()

,total_cases,total_deaths,new_cases,new_deaths,new_cases_smoothed,new_deaths_smoothed,new_cases_per_million,total_cases_per_million,new_cases_smoothed_per_million,new_deaths_per_million,total_deaths_per_million,new_deaths_smoothed_per_million
count,33669.000000,33663.000000,33663.000000,33666.000000,33669.000000,13391.000000,13494.000000,13484.000000,13408.000000,13606.000000,33669.000000,33669.000000
mean,1938.812944,1.708879,7.784422,0.012297,7.780489,0.012493,0.085680,19.413230,0.072734,0.000068,0.011201,0.000080
std,8458.943848,8.498339,63.688841,0.216703,49.287428,0.089431,1.046114,30.643406,0.313962,0.002010,0.041833,0.000991
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.688500,0.000000,0.000000,0.000000,0.000000
50%,21.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.878000,0.000000,0.000000,0.000000,0.000000
75%,257.000000,0.000000,0.000000,0.000000,0.570000,0.000000,0.000000,28.180000,0.024000,0.000000,0.000000,0.000000
max,87376.000000,140.000000,1802.000000,12.000000,1089.140000,1.710000,82.212000,183.615000,17.443000,0.111100,0.587380,0.031760


### 3. Nettoyage  des colonnes inutile



In [44]:
df_clean = df_raw.copy()

cols_to_drop = ['iso_code']

df_clean = df_clean.drop(
    columns=[col for col in cols_to_drop if col in df_clean.columns]
)

df_clean.head()

,location,date,total_cases,total_deaths,new_cases,new_deaths,new_cases_smoothed,new_deaths_smoothed,new_cases_per_million,total_cases_per_million,new_cases_smoothed_per_million,new_deaths_per_million,total_deaths_per_million,new_deaths_smoothed_per_million
0,Africa,2022-05-01,27.0,2.0,0.0,0.0,0.29,NaN,NaN,NaN,NaN,NaN,0.0014,0.0
1,Africa,2022-05-02,27.0,2.0,0.0,0.0,0.29,NaN,NaN,0.019,NaN,0.0,0.0014,0.0
2,Africa,2022-05-03,27.0,2.0,0.0,0.0,0.29,0.0,NaN,NaN,0.0,NaN,0.0014,0.0
3,Africa,2022-05-04,27.0,2.0,0.0,0.0,0.29,NaN,0.0,0.019,NaN,0.0,0.0014,0.0
4,Africa,2022-05-05,27.0,2.0,0.0,0.0,0.29,NaN,0.0,NaN,NaN,NaN,0.0014,0.0


### 4. Renommage des colonnes



In [45]:
df_clean = df_clean.rename(columns={
    'location': 'country',
    'new_cases': 'daily_new_cases',
    'new_deaths': 'daily_new_deaths',
    'total_cases': 'total_cases',
    'total_deaths': 'total_deaths'
})

df_clean.head()

,country,date,total_cases,total_deaths,daily_new_cases,daily_new_deaths,new_cases_smoothed,new_deaths_smoothed,new_cases_per_million,total_cases_per_million,new_cases_smoothed_per_million,new_deaths_per_million,total_deaths_per_million,new_deaths_smoothed_per_million
0,Africa,2022-05-01,27.0,2.0,0.0,0.0,0.29,NaN,NaN,NaN,NaN,NaN,0.0014,0.0
1,Africa,2022-05-02,27.0,2.0,0.0,0.0,0.29,NaN,NaN,0.019,NaN,0.0,0.0014,0.0
2,Africa,2022-05-03,27.0,2.0,0.0,0.0,0.29,0.0,NaN,NaN,0.0,NaN,0.0014,0.0
3,Africa,2022-05-04,27.0,2.0,0.0,0.0,0.29,NaN,0.0,0.019,NaN,0.0,0.0014,0.0
4,Africa,2022-05-05,27.0,2.0,0.0,0.0,0.29,NaN,0.0,NaN,NaN,NaN,0.0014,0.0


### 5. Nettoyage de la date

La colonne `date` a été convertie au format datetime afin de permettre des analyses temporelles : évolution des cas, agrégation par mois, suivi de la progression de l’épidémie.

In [46]:
df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')

print(df_clean['date'].isnull().sum())
df_clean[['date']].head()

0


,date
0,2022-05-01
1,2022-05-02
2,2022-05-03
3,2022-05-04
4,2022-05-05


### 6. Suppression des agrégats non-pays
Les lignes correspondant aux continents ou aux agrégats globaux ont été supprimées afin de conserver uniquement les observations par pays.

In [47]:
countries_to_exclude = [
    "Africa", "Europe", "North America", "South America",
    "Asia", "World", "Oceania", "Puerto Rico"
]

df_clean = df_clean[~df_clean['country'].isin(countries_to_exclude)]

df_clean['country'].unique()[:20]

array(['Andorra', 'Argentina', 'Aruba', 'Australia', 'Austria', 'Bahamas',
       'Bahrain', 'Barbados', 'Belgium', 'Benin', 'Bermuda', 'Bolivia',
       'Bosnia and Herzegovina', 'Brazil', 'Bulgaria', 'Cameroon',
       'Canada', 'Central African Republic', 'Chile', 'China'],
      dtype=object)

### 7. Suppression des colonnes redondantes


In [48]:
redundant_cols = [
    'new_cases_smoothed',
    'new_deaths_smoothed',
    'new_cases_per_million',
    'total_cases_per_million',
    'new_cases_smoothed_per_million',
    'new_deaths_per_million',
    'total_deaths_per_million',
    'new_deaths_smoothed_per_million'
]

existing_redundant = [col for col in redundant_cols if col in df_clean.columns]

df_clean = df_clean.drop(columns=existing_redundant)

print("Colonnes supprimées :", existing_redundant)
df_clean.head()

Colonnes supprimées : ['new_cases_smoothed', 'new_deaths_smoothed', 'new_cases_per_million', 'total_cases_per_million', 'new_cases_smoothed_per_million', 'new_deaths_per_million', 'total_deaths_per_million', 'new_deaths_smoothed_per_million']


,country,date,total_cases,total_deaths,daily_new_cases,daily_new_deaths
370,Andorra,2022-07-25,2.0,0.0,2.0,0.0
371,Andorra,2022-07-26,3.0,0.0,1.0,0.0
372,Andorra,2022-07-27,3.0,0.0,0.0,0.0
373,Andorra,2022-07-28,3.0,0.0,0.0,0.0
374,Andorra,2022-07-29,3.0,0.0,0.0,0.0


### 8. Gestion des valeurs manquantes
Les valeurs manquantes des colonnes numériques principales ont été imputées par la médiane. Ce choix permet de limiter l’influence des valeurs extrêmes, fréquentes dans les données épidémiologiques où certains pays peuvent concentrer un nombre de cas beaucoup plus élevé que d’autres.

In [51]:
# Colonnes à imputer
cols_to_impute = [
    "total_deaths",
    "daily_new_cases",
    "daily_new_deaths"
]

# Conversion de la date
df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")

# Création d'une colonne mois
df_clean["year_month"] = df_clean["date"].dt.to_period("M")

# Imputation par médiane mensuelle par pays, puis fallback
for col in cols_to_impute:
    if col in df_clean.columns:
        # 1. Médiane par pays et par mois
        df_clean[col] = df_clean.groupby(["country", "year_month"])[col].transform(
            lambda x: x.fillna(x.median())
        )

df_clean = df_clean.drop(columns=["year_month"])
# Vérification
print("Valeurs manquantes après imputation :")
print(df_clean[cols_to_impute].isnull().sum())

Valeurs manquantes après imputation :
total_deaths        0
daily_new_cases     0
daily_new_deaths    0
dtype: int64


### 9. tri, suppression des doublons et validation finale


In [54]:
# Suppression des doublons
df_clean = df_clean.drop_duplicates()

# Tri par pays puis par date
df_clean = df_clean.sort_values(by=['country', 'date'])

# Réinitialisation de l'index
df_clean = df_clean.reset_index(drop=True)

df_clean.head()

,country,date,total_cases,total_deaths,daily_new_cases,daily_new_deaths
0,Andorra,2022-07-25,2.0,0.0,2.0,0.0
1,Andorra,2022-07-26,3.0,0.0,1.0,0.0
2,Andorra,2022-07-27,3.0,0.0,0.0,0.0
3,Andorra,2022-07-28,3.0,0.0,0.0,0.0
4,Andorra,2022-07-29,3.0,0.0,0.0,0.0


In [55]:
print("Dimensions finales :", df_clean.shape)
print("Doublons restants :", df_clean.duplicated().sum())
print("Valeurs manquantes restantes :")
df_clean.isnull().sum()

Dimensions finales : (30836, 6)
Doublons restants : 0
Valeurs manquantes restantes :


country             0
date                0
total_cases         0
total_deaths        0
daily_new_cases     0
daily_new_deaths    0
dtype: int64

### 10. Sauvegarde des données propres
Pour ce projet, le travail de data wrangling comprend :

- chargement du CSV brut ;
- audit initial du dataset : dimensions, types, valeurs manquantes et doublons ;
- conversion de la colonne date au format datetime ;
- suppression des agrégats globaux et continentaux ;
- renommage des colonnes principales ;
- traitement des valeurs manquantes par imputation ;
- suppression des colonnes redondantes ou peu utiles pour l’analyse ;
- vérification et suppression des doublons ;
- tri des données par pays et par date ;
- sauvegarde du dataset nettoyé pour l’analyse exploratoire.

In [57]:
processed_path = "../data/processed/owid-monkeypox-data_clean.csv"

df_clean.to_csv(processed_path, index=False)

print(f"Données nettoyées sauvegardées dans : {processed_path}")

Données nettoyées sauvegardées dans : ../data/processed/owid-monkeypox-data_clean.csv
